In [0]:
# Silver Layer: Returns — Cleansing + CDC MERGE

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, trim, upper, lower, try_to_date, to_timestamp,
    when, coalesce, lit, current_timestamp,
    regexp_replace, row_number, desc
)
from pyspark.sql.types import DoubleType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()

BRONZE_TABLE = "ecomstore.ecomlake.bronze_returns"
SILVER_TABLE = "ecomstore.ecomlake.silver_returns"
TEMP_STAGING_TABLE = "ecomstore.ecomlake.temp_returns_staging"

# Read from Bronze 
bronze_df = spark.read.table(BRONZE_TABLE)

# Deduplication 
# Keep latest record per return_id
w = Window.partitionBy("return_id").orderBy(desc("_ingestion_timestamp"))

deduped_df = (
    bronze_df
    .withColumn("rn", row_number().over(w))
    .filter(col("rn") == 1)
    .drop("rn")
)

# Cleansing & Standardization
cleaned_df = (
    deduped_df
    # key fields
    .withColumn("return_id",     trim(upper(col("return_id"))))
    .withColumn("order_id",      trim(upper(col("order_id"))))
    .withColumn("order_item_id", trim(upper(col("order_item_id"))))

    # return_reason: normalize case inconsistency
    .withColumn("return_reason",
        when(col("return_reason").isNotNull(), lower(trim(col("return_reason"))))
        .otherwise(lit("unspecified"))
    )
    .withColumn("return_reason",
        when(col("return_reason").isin(
            "damaged", "wrong_item", "not_as_described", "changed_mind", "unspecified"),
             col("return_reason"))
        .otherwise(lit("other"))
    )

    # return_status: normalize to valid set
    .withColumn("return_status",
        when(col("return_status").isin(
            "initiated", "picked_up", "received", "refund_processed", "rejected"),
             col("return_status"))
        .otherwise(lit("initiated"))
    )

    # refund_amount: strip currency symbols, null out negatives
    .withColumn("refund_amount",
        regexp_replace(col("refund_amount").cast("string"), r"[^\d.]", "").cast(DoubleType())
    )
    .withColumn("refund_amount",
        when(col("refund_amount") < 0, None).otherwise(col("refund_amount"))
    )

    # date fields: handle corrupted formats
    .withColumn("return_date",
        coalesce(
            try_to_date(col("return_date"), "yyyy-MM-dd"),
            try_to_date(col("return_date"), "dd/MM/yyyy")
        )
    )
    .withColumn("refund_date",
        coalesce(
            try_to_date(col("refund_date"), "yyyy-MM-dd"),
            try_to_date(col("refund_date"), "dd/MM/yyyy")
        )
    )

    # timestamps
    .withColumn("created_at",
        coalesce(to_timestamp(col("created_at")), to_timestamp(col("created_at"), "yyyy-MM-dd HH:mm:ss")))

    # metadata
    .withColumn("_silver_processed_at", current_timestamp())

    .filter(col("return_id").isNotNull() & col("order_id").isNotNull())
)

# Final Silver schema
silver_df = cleaned_df.select(
    "return_id", "order_id", "order_item_id",
    "return_reason", "return_status",
    "refund_amount", "return_date", "refund_date",
    "created_at",
    "_batch_id", "_silver_processed_at"
)

# Disk Staging to prevent Double Compute
silver_df.write.format("delta").mode("overwrite").saveAsTable(TEMP_STAGING_TABLE)
staged_silver_df = spark.read.table(TEMP_STAGING_TABLE)

# Data Quality split
# Apply filters to the staged data
good_records = staged_silver_df.filter(
    col("return_id").isNotNull() &
    col("order_id").isNotNull() &
    col("return_date").isNotNull()
)

bad_records = staged_silver_df.filter(
    col("return_id").isNull() |
    col("order_id").isNull() |
    col("return_date").isNull()
)

(
    bad_records
    .withColumn("rejection_reason",
        when(col("return_id").isNull(),   lit("null_return_id"))
        .when(col("order_id").isNull(),   lit("null_order_id"))
        .otherwise(lit("null_return_date"))
    )
    .write.format("delta").mode("append")
    .option("mergeSchema", "true")
    .saveAsTable("ecomstore.ecomlake.silver_quarantine_returns")
)

# CDC MERGE
if spark.catalog.tableExists(SILVER_TABLE):
    silver_target = DeltaTable.forName(spark, SILVER_TABLE)

    (
        silver_target.alias("target")
        .merge(
            good_records.alias("source"),
            "target.return_id = source.return_id"
        )
        # UPDATE when return_status progressed or refund details arrived
        .whenMatchedUpdate(
            condition="source.return_status != target.return_status OR "
                      "(source.refund_date IS NOT NULL AND target.refund_date IS NULL)",
            set={
                "return_status":        "source.return_status",
                "refund_amount":        "source.refund_amount",
                "refund_date":          "source.refund_date",
                "_batch_id":            "source._batch_id",
                "_silver_processed_at": "source._silver_processed_at"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )
    print(f"✅ CDC MERGE complete on {SILVER_TABLE}")

else:
    (
        good_records
        .write
        .format("delta")
        .mode("overwrite")
        .option("delta.autoOptimize.optimizeWrite", "true")
        .option("delta.autoOptimize.autoCompact", "true") 
        .saveAsTable(SILVER_TABLE)
    )
    print(f"✅ Silver returns table created: {SILVER_TABLE}")

# Clean up staging table
spark.sql(f"DROP TABLE IF EXISTS {TEMP_STAGING_TABLE}")